# 07 — Monte Carlo Playoff Simulation

Simulate the MLS Cup Playoffs using Monte Carlo methods:
- Remaining regular season simulation
- Best-of-3 playoff bracket
- Conference and MLS Cup champion probabilities

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
import warnings
warnings.filterwarnings('ignore')

from mls_predictor.data_loader import load_raw_data
from mls_predictor.elo import compute_elo_history, get_current_elo
from mls_predictor.monte_carlo import (
    compute_current_standings, simulate_playoff_bracket,
)
from mls_predictor.config import (
    EASTERN_CONFERENCE, WESTERN_CONFERENCE, MC_SIMULATIONS,
)

In [ ]:
df = load_raw_data()
df = compute_elo_history(df)
current_season = int(df['Season'].max())
current_elo = get_current_elo(df)
standings = compute_current_standings(df, current_season)

print(f'Season: {current_season}')
print(f'Teams with standings: {len(standings)}')

In [ ]:
# Simulate Eastern Conference Playoffs
n_sims = 10000

east_standings = [
    (t, standings.get(t, {'points': 0, 'gd': 0}))
    for t in EASTERN_CONFERENCE if t in standings
]
east_standings.sort(key=lambda x: (x[1]['points'], x[1]['gd']), reverse=True)
east_seeded = [t for t, _ in east_standings]

print('Eastern Conference Seeding:')
for i, (t, s) in enumerate(east_standings[:9], 1):
    print(f'  {i}. {t:25s} {s["points"]:2d} pts  (Elo: {current_elo.get(t, 1500):.0f})')

east_results = simulate_playoff_bracket(east_seeded, current_elo, n_sims=n_sims)
east_results

In [ ]:
# Simulate Western Conference Playoffs
west_standings = [
    (t, standings.get(t, {'points': 0, 'gd': 0}))
    for t in WESTERN_CONFERENCE if t in standings
]
west_standings.sort(key=lambda x: (x[1]['points'], x[1]['gd']), reverse=True)
west_seeded = [t for t, _ in west_standings]

print('Western Conference Seeding:')
for i, (t, s) in enumerate(west_standings[:9], 1):
    print(f'  {i}. {t:25s} {s["points"]:2d} pts  (Elo: {current_elo.get(t, 1500):.0f})')

west_results = simulate_playoff_bracket(west_seeded, current_elo, n_sims=n_sims)
west_results

In [ ]:
# Combined visualization
all_results = pd.concat([
    east_results.assign(conference='Eastern'),
    west_results.assign(conference='Western'),
]).sort_values('champion_pct', ascending=False)

top15 = all_results.head(15)

fig = px.bar(
    top15, x='champion_pct', y='team',
    color='conference', orientation='h',
    color_discrete_map={'Eastern': '#00d4ff', 'Western': '#ff4757'},
    title='Top 15 MLS Cup Contenders (10K Simulations)',
    labels={'champion_pct': 'Conference Champion %', 'team': ''},
)
fig.update_layout(
    template='plotly_dark',
    yaxis=dict(autorange='reversed'),
    height=500,
)
fig.show()

In [ ]:
# Full bracket probabilities
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

for ax, results, conf_name, color in [
    (axes[0], east_results, 'Eastern', '#00d4ff'),
    (axes[1], west_results, 'Western', '#ff4757'),
]:
    r = results.head(9)
    x = np.arange(len(r))
    w = 0.25
    
    ax.barh(x - w, r['semifinal_pct'], w, label='Semifinal', alpha=0.4, color=color)
    ax.barh(x, r['conf_final_pct'], w, label='Conf Final', alpha=0.7, color=color)
    ax.barh(x + w, r['champion_pct'], w, label='Champion', alpha=1.0, color=color)
    
    ax.set_yticks(x)
    ax.set_yticklabels(r['team'])
    ax.set_xlabel('Probability (%)')
    ax.set_title(f'{conf_name} Conference', fontweight='bold')
    ax.legend()
    ax.invert_yaxis()

plt.suptitle(f'MLS {current_season} Playoff Probabilities ({n_sims:,} simulations)',
             fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()